# Lifetimes and mixing

Besides four-momenta, the decay tree contains the information that is needed to simulate decays in flight: production vertices, sampled lifetimes, and the decay mode. Neutral $B$ mesons also mix, which shows up as an extra node in the tree.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib_inline.backend_inline import set_matplotlib_formats

set_matplotlib_formats("svg")

Creating a {py:class}`.Generator` reads the decay and particle tables and initializes Pythia8, which prints a lot of log output. That output is hidden on this page.

In [ ]:
from pyevtgen import Generator

generator = Generator(seed=0)
events = generator.generate("B0", n=20_000)

## Lifetimes

Each unstable particle has a sampled proper decay time {py:attr}`~.Particle.lifetime`, given as $c\tau$ in mm. Its distribution is exponential with the mean $c\tau$ from the particle table:

In [ ]:
ctau = np.array([event.lifetime for event in events])
expected = generator.properties("B0").ctau
print(f"sampled mean: {ctau.mean():.4f} mm, table: {expected:.4f} mm")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(ctau, bins=60, density=True, label="generated")
x = np.linspace(0, ctau.max(), num=200)
ax.plot(x, np.exp(-x / expected) / expected, label="exponential")
ax.set_xlabel(r"$c\tau$ [mm]")
ax.set_yscale("log")
ax.legend()
plt.show()

## Production vertices

The {py:attr}`~.Particle.position` of a particle is its production four-position $(ct, x, y, z)$ in mm. The daughters of a parent that decays at rest are produced at the same point in space, but later in time:

In [ ]:
event = events[1]
print("parent:  ", event.position)
print("daughter:", event.daughters[0].position)

For a parent in flight, the decay vertex moves along the direction of flight:

In [ ]:
mass = generator.properties("B0").mass
p4 = [10.0, 0.0, 0.0, np.sqrt(10.0**2 - mass**2)]
event = generator.decay("B0", p4=p4)
event.daughters[0].position

## Decay modes

{py:attr}`~.Particle.decay_mode` is the index of the decay channel that EvtGen selected from the decay table of the particle. For a $D^+$, each index corresponds to one set of daughters:

In [ ]:
from collections import Counter, defaultdict

d_events = generator.generate("D+", n=20_000)
channels = defaultdict(set)
for event in d_events:
    channels[event.decay_mode].add(" ".join(d.name for d in event.daughters))
for mode, count in Counter(e.decay_mode for e in d_events).most_common(4):
    print(f"{mode:>3}: {count:>5} x {sorted(channels[mode])}")

:::{caution}
EvtGen delegates many inclusive $B$ meson decays to Pythia8 through the `PYTHIA` decay model. Pythia8 resets the selected channel when it replaces the daughters, so all of those decays report {py:attr}`~.Particle.decay_mode` 0, whatever their daughters are:
:::

In [ ]:
b_channels = {
    " ".join(d.name for d in event.daughters)
    for event in events
    if event.decay_mode == 0 and event.daughters[0].name != "anti-B0"
}
len(b_channels)

## Mixing

With the default incoherent mixing (`mixing_type=1`, see {py:class}`.Generator`), a neutral $B^0$ that oscillates before it decays gets a single daughter: an $\bar{B}^0$ that then decays. The fraction of such events gives the time-integrated mixing probability $\chi_d$, which is about 0.19:

In [ ]:
mixed = [event for event in events if event.daughters[0].name == "anti-B0"]
len(mixed) / len(events)

In [ ]:
print([daughter.name for daughter in mixed[0].daughters])

Coherent mixing (`mixing_type=0`) is meant for $B$ meson pairs from $\Upsilon(4S)$ decays. Because a process can hold only one {py:class}`.Generator`, the mixing type has to be chosen when the generator is created.